In [1]:
import pickle

import numpy as np 
import re 
from pathlib import Path
import pandas as pd
import json
import pickle
import importlib 
import IPython.display as ipd
%matplotlib inline 

import matplotlib.pyplot as plt 
import seaborn as sns
from scipy import stats 
# from matplotlib.ticker import FormatStrFormatter
from src import util_analysis
import src.util_process_prolific as up 
from tqdm import tqdm
importlib.reload(up)

<module 'src.util_process_prolific' from '/net/vast-storage/scratch/vast/mcdermott/imgriff/projects/torch_2_aud_attn/src/util_process_prolific.py'>

# Experiment 1

In [2]:
## import class maps
import pickle
cv_word_2_class = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 
cv_class_2_word = {v:k for k,v in cv_word_2_class.items()}

### Get conditions 
path_to_expmt_stim = Path("/om/user/imgriff/datasets/human_word_rec_SWC_2024/")
manifests = pd.read_pickle(path_to_expmt_stim / "human_cue_target_distractor_df_w_meta_transcripts_w_f0.pdpkl")


cond_map = pickle.load( open(path_to_expmt_stim / "human_attn_expmt_cond_map.pkl", "rb" )) 
conds = set([cond for cond,snr in cond_map.values()])

test_condition_dict = {'music':"background_musdb18hq",
                       "babble":"background_cv08talkerbabble",
                       "stationary": "background_issnstationary",
                       "natural scene": "background_ieeeaaspcasa",
                       "clean": "SILENCE"}

test_cond_to_human = {v:k for k,v in test_condition_dict.items()}
cond_remap = {}
for cond in conds:
    if cond in test_cond_to_human:
        cond_remap[cond] = test_cond_to_human[cond]
    else:
        cond_remap[cond] = cond 
cond_remap['natural scene'] = 'natural scene'


In [ ]:
def remap_1_distractor_str(cond_str):
    if 'english' in cond_str:
        return '1-talker'
    if 'mandarin' in cond_str:
        return "Mandarin distractor"
    if 'dutch' in cond_str:
        return "Dutch distractor"

In [4]:

model_names = [ ]

# get names for arch_search models 
arch_search_models = [path.stem for path in Path('swc_2024_eval_full_stim/').glob('*post*')]
model_names.extend(arch_search_models)

model_name_dict = {
                   "word_task_v10_gain_post_norm_config": 'Norm before gain model',
                  }


all_model_results = []
all_model_f0_results = []

str_to_cond = {v:k for k,v in test_condition_dict.items()}

## preselect manifest columns 
cols_to_merge = [
                #  'stim_name',
                 'gender',
                 'word',
                 'word_int',
                 'target_f0',
                 'english_distractor_f0',
                 'mandarin_distractor_f0',
                 'dutch_distractor_f0',
                 'target_transcripts',
                 'same_sex_distractor_1_transcripts',
                 'diff_sex_distractor_1_transcripts',
                 "same_sex_dist_1_word",
                 "diff_sex_dist_1_word"

]

manifests['word_int'] = manifests.word.replace(cv_word_2_class)
model_manifest = manifests[cols_to_merge].sort_values(['gender', 'word']).copy()
model_manifest['df_index'] = model_manifest.index

# update gt manifests to match readable format 
for model_name in model_names:
    print(model_name)
    output_paths = list(Path(f'swc_2024_eval_full_stim/{model_name}').glob('*.csv'))
    print(len(output_paths))
    results_dfs = []

    for path in output_paths:
        # print(path)
        try:
            df = pd.read_csv(path)
            # reformat dict
            df['model'] = path.parent.name
            #parts of name 
            parts = path.stem.split(path.parent.name)[-1].split('_')   
            # if 'stim_tag' in df.columns:
            #     # rename 
            #     df.rename(columns={'stim_tag':'stim_name'}, inplace=True)
            # use re to split path.stem after model name and before <int>dB 
            df['background_condition'] = [v for v in cond_remap.values() if "_".join(v.split(' ')) in path.stem][0]
            if 'clean' in path.stem:
                df['background_condition'] = 'clean'
                df['snr'] =  'inf' # really np.inf, 6 for plotting 
            else:
                df['snr'] = int(re.search('(-?\d+)dB', path.stem).group(0).strip('dB'))

            df['test_index'] = df.index
            df = pd.merge(df,
                model_manifest,
                            left_on=["test_index", "true_word_int"], right_on=["df_index", "word_int"])
            if 'dutch' in df['background_condition'].unique()[0]:
                continue
            results_dfs.append(df)
        except Exception as e:
            # print(e)
            continue

    model_results = pd.concat(results_dfs, axis=0, ignore_index=True)
    if model_name in model_name_dict:
        model_str = model_name_dict[model_name]
    elif "arch_" in model_name:
        model_str = "Feature-gain Model"
    else:
        model_str = model_name

    model_results['group'] = model_str
    ## Load in model vocab 
    model_results['pred_word'] = model_results['pred_word_int'].replace(cv_class_2_word)
    model_results['true_word'] = model_results['true_word_int'].replace(cv_class_2_word)


    # Add 1-talker condition metadata to model results

    # add confusions 
    model_results['confusions'] = (model_results.pred_word == model_results.same_sex_dist_1_word).astype('int')

    # add adjusted accuracy and confusions 
    pred_words = model_results.pred_word.values
    target_words = model_results.word.values
    target_transcripts = model_results.target_transcripts.values
    same_sex_distractor_words = model_results.same_sex_dist_1_word.values
    diff_sex_distractor_words = model_results.same_sex_dist_1_word.values
    same_sex_distractor_transcripts = model_results.same_sex_distractor_1_transcripts.values
    diff_sex_distractor_transcripts = model_results.diff_sex_distractor_1_transcripts.values


    adjusted_acc = np.array([int(pred_word in target_transcript or pred_word == target_word)
                                if not isinstance(target_transcript, float) else np.nan
                                for pred_word, target_word, target_transcript in zip(pred_words, target_words, target_transcripts)
                                ])

    adjusted_confs = np.array([int(pred_word in same_sex_transcript or pred_word in diff_sex_transcript or pred_word == same_sex_word or pred_word == diff_sex_word)
                                if not (isinstance(same_sex_transcript, float) and isinstance(diff_sex_transcript, float)) else np.nan
                                for pred_word, same_sex_word, diff_sex_word, same_sex_transcript, diff_sex_transcript in zip(pred_words, same_sex_distractor_words, diff_sex_distractor_words,  same_sex_distractor_transcripts, diff_sex_distractor_transcripts)
                                ])
    
    model_results['adjusted_accuracy'] = adjusted_acc
    model_results['adjusted_confusions'] = adjusted_confs
    model_results.loc[model_results.background_condition.str.contains('1-talker'), 'sex_cond'] = model_results.loc[model_results.background_condition.str.contains('1-talker'), 'background_condition'].str.split('-').str[-1].str.title()
    model_results.loc[model_results.background_condition.str.contains('1-talker'), 'dist_lang'] = model_results.loc[model_results.background_condition.str.contains('1-talker'), 'background_condition'].str.split('-').str[-2].str.title()
    model_results.loc[model_results.background_condition.str.contains('1-talker'), 'background_condition'] = model_results.loc[model_results.background_condition.str.contains('1-talker'), 'background_condition'].apply(remap_1_distractor_str)

    
    all_model_results.append(model_results)


    model_f0_df = model_results[model_results.background_condition.isin([ "1-talker", 'Mandarin distractor', 'Dutch distractor', 'clean'])].copy()
    model_f0_df.loc[model_f0_df.dist_lang == 'English', 'distractor_f0'] = model_f0_df.loc[model_f0_df.dist_lang == 'English', 'english_distractor_f0'].values
    model_f0_df.loc[model_f0_df.dist_lang == 'Mandarin', 'distractor_f0'] = model_f0_df.loc[model_f0_df.dist_lang == 'Mandarin', 'mandarin_distractor_f0'].values
    model_f0_df.loc[model_f0_df.dist_lang == 'Dutch', 'distractor_f0'] = model_f0_df.loc[model_f0_df.dist_lang == 'Dutch', 'dutch_distractor_f0'].values
    model_f0_df["abs_f0_diff"] = np.abs(model_f0_df.target_f0 - model_f0_df.distractor_f0)
    model_f0_df["percent_f0_diff"] = model_f0_df.distractor_f0 / model_f0_df.target_f0 
    # model_f0_df["f0_ratio"] = model_f0_df.apply(get_f0_ratio, axis=1)
    model_f0_df.loc[:, "abs_f0_diff"] = np.abs(model_f0_df.target_f0 - model_f0_df.distractor_f0)
    model_f0_df.loc[:, "percent_f0_diff"] = model_f0_df.distractor_f0 / model_f0_df.target_f0 
    all_model_f0_results.append(model_f0_df)

all_model_results = pd.concat(all_model_results, axis=0, ignore_index=True)
all_model_f0_results = pd.concat(all_model_f0_results, axis=0, ignore_index=True)

/tmp/ipykernel_456881/543971170.py:35: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  manifests['word_int'] = manifests.word.replace(cv_word_2_class)


word_task_v10_gain_post_norm_config
61


In [ ]:
all_model_results.model.value_counts()


model
word_task_v10_gain_post_norm_config    49776
Name: count, dtype: int64

In [ ]:
all_model_results.loc[all_model_results.snr == np.inf, 'snr'] = 6
all_model_results.loc[all_model_results.snr == 'inf', 'snr'] = 6
all_model_results.snr = all_model_results.snr.astype('int')

all_model_results['background_condition'].rename({"stationary":"noise"}, inplace=True)
# all_model_results = all_model_results[all_model_results.snr > -12]
all_model_results.loc[all_model_results.background_condition == "stationary", 'background_condition'] = "noise"

In [7]:
#################################
# Setup summary for models  
################################# 
all_model_f0_results_to_analyze = all_model_f0_results.copy()
all_model_f0_results_to_analyze['id_subject'] = all_model_f0_results_to_analyze['group']
all_model_f0_results_to_analyze.loc[all_model_f0_results_to_analyze.snr == 'inf', 'sex_cond'] = 'clean'
all_model_f0_results_to_analyze.loc[all_model_f0_results_to_analyze.snr == 'inf', 'dist_lang'] = 'clean'

model_f0_summary = (all_model_f0_results_to_analyze.groupby(["snr", "model", "sex_cond", "dist_lang"])
                     .agg({
                            'adjusted_accuracy':['mean'],
                            'adjusted_confusions':['mean']})
                     .reset_index())
model_f0_summary['snr'] = model_f0_summary['snr'].replace(np.inf, 6)
model_f0_summary['snr'] =  model_f0_summary['snr'].replace("inf", 6)
model_f0_summary['snr'] = model_f0_summary['snr'].astype('int')
# flatten multiindex 
model_f0_summary.columns = ['_'.join(col).strip() for col in model_f0_summary.columns.values]
# remove trailing underscore
model_f0_summary.columns = [col[:-1] if col.endswith('_') else col for col in model_f0_summary.columns.values]



#########################################
# Summarize sex and lang results models
#########################################
control_model_f0_df = model_f0_summary[~model_f0_summary.model.str.contains('main|arch_')].copy()
control_sex_summary = []
for (model, snr, sex_cond), _ in control_model_f0_df.groupby(['model', 'snr', 'sex_cond']):
       if sex_cond == 'clean':
              continue
       else:
              data = all_model_f0_results_to_analyze[(all_model_f0_results_to_analyze.model == model)
                                                        & (all_model_f0_results_to_analyze.snr == snr)
                                                        & (all_model_f0_results_to_analyze.sex_cond == sex_cond)]
              accs = data.adjusted_accuracy.values
              acc_sem = util_analysis.bootstrap_sem(accs)
              group = data.group.values[0]

              # get confusions
              confs = data[data.dist_lang == 'English'].adjusted_confusions.values     
              conf_sem = util_analysis.bootstrap_sem(confs)

              control_sex_summary.append(
                                   {'group':group, 'model':model, 'snr': snr, "background_condition":sex_cond, 
                                   'accuracy': accs.mean(), 'acc_sem': acc_sem,
                                   'confusions': confs.mean(), 'conf_sem': conf_sem}
                     )

control_sex_summary_df = pd.DataFrame(control_sex_summary)

control_lang_summary = []
for (model, snr, dist_lang), _ in control_model_f0_df.groupby(['model', 'snr','dist_lang']):
       if dist_lang == 'clean':
              continue 

       else:
              data = all_model_f0_results_to_analyze[(all_model_f0_results_to_analyze.model == model)
                                                        & (all_model_f0_results_to_analyze.snr == snr)
                                                        & (all_model_f0_results_to_analyze.dist_lang == dist_lang)]
              accs = data.adjusted_accuracy.values
              confs = data.adjusted_confusions.values
              acc_sem = util_analysis.bootstrap_sem(accs)
              conf_sem = util_analysis.bootstrap_sem(confs)
              group = data.group.values[0]

              control_lang_summary.append(
                                   {'group':group, 'model':model, 'snr': snr, "background_condition":dist_lang,
                                   'accuracy': accs.mean(), 'acc_sem': acc_sem,
                                   'confusions': confs.mean(), 'conf_sem': conf_sem}
                     )

control_lang_summary_df = pd.DataFrame(control_lang_summary)
control_sex_lang_summary = pd.concat([control_sex_summary_df, control_lang_summary_df], axis=0, ignore_index=True).drop_duplicates()


##########################
# set common column names 
##########################
full_df_col_names = ['model', 'snr', 'background_condition', 'accuracy', 'acc_sem', 'confusions', 'conf_sem', 'N']
summary_df_col_names = ['group', 'snr', 'background_condition', 'accuracy', 'acc_sem', 'confusions', 'conf_sem', 'N']

# screen language results - will use what we summarized above
combined_results = all_model_results[~all_model_results.background_condition.str.contains('Mandarin|Dutch')]


#############################
# Get control model summary
#############################
control_list = 'Norm'

control_models = combined_results[combined_results.group.str.contains(control_list)].copy()
control_summary = []
for (model, snr, condition), group in control_models.groupby(['model', 'snr', 'background_condition']):
    accs = group.adjusted_accuracy.values
    confs = group.adjusted_confusions.values
    acc_sem = util_analysis.bootstrap_sem(accs)
    conf_sem = util_analysis.bootstrap_sem(confs)
    group = group.group.values[0]
    control_summary.append(
                            {'group':group, 'model':model, 'snr': snr, 'background_condition': condition, 
                             'accuracy': accs.mean(), 'acc_sem': acc_sem,
                             'confusions': confs.mean(), 'conf_sem': conf_sem, 'N':1}
              )
control_summary = pd.DataFrame(control_summary)
# merge and drop duplicate clean row from distractor language analysis 
exp_1_df = pd.concat([control_summary, control_sex_lang_summary], axis=0, ignore_index=True).drop_duplicates()


/tmp/ipykernel_456881/2225343903.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  model_f0_summary['snr'] =  model_f0_summary['snr'].replace("inf", 6)


In [ ]:
exp_1_df.model.value_counts()

model
word_task_v10_gain_post_norm_config    56
Name: count, dtype: int64

In [ ]:
exp_1_df['snr'] = exp_1_df['snr'].astype('int')
exp_1_df['model'] = exp_1_df['model'].astype('str')

## Experiment 2

In [10]:
meta_df = pd.read_pickle('/om/user/imgriff/datasets/human_swc_popham_exmpt_2024/source_stim_meta_manifest.pdpkl')
## import class maps
import pickle
## load WSN vocab mapping 
word_and_speaker_encodings = pickle.load( open( "/om2/user/imgriff/projects/Auditory-Attention/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
wsn_word_2_class = word_and_speaker_encodings['word_to_idx']
wsn_class_2_word = word_and_speaker_encodings['word_idx_to_word']
cv_word_2_class = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 
cv_class_2_word = {v:k for k,v in cv_word_2_class.items()}

# parent_path = Path('/om2/user/imgriff/projects/torch_2_aud_attn/popham_mono_eval/')
parent_path = Path('/om2/user/imgriff/projects/torch_2_aud_attn/popham_swc_eval_all_stim/')

# model_name = 'word_task_standard_v08'
# model_name = 'word_task_half_co_loc_v07'

meta_df = pd.read_pickle('/om/user/imgriff/datasets/human_word_rec_SWC_2024/human_cue_target_distractor_df_w_meta_transcripts.pdpkl')
meta_df['expt_df_ix'] = meta_df.index
cols_to_merge = [
                 'gender',
                 'word',
                 'target_transcripts',
                 'same_sex_distractor_1_transcripts',
                 'diff_sex_distractor_1_transcripts',
                 "same_sex_dist_1_word",
                 "diff_sex_dist_1_word",
                 "expt_df_ix"
]

results_dirs = list(parent_path.rglob(f"*/*post*.csv"))
print(len(results_dirs))

dfs = []

for result_csv in results_dirs:
    # print(result_csv)
    # if not 'word_task_half_co_loc_v09_gender_bal_4M_w_no_cue_learned_higher_lr_less_dropout' in str(result_csv) or not '_arch_' in str(result_csv):
    #     # print('skp')
    #     continue
    try:
        df = pd.read_csv(result_csv)
    except Exception as e:
        print(e)
        print(result_csv)
        continue
    
    # break 
    df['target_harmonicity'] = result_csv.stem.split('_target')[0].split('_')[-1].title()
    df['model'] = result_csv.parent.stem
    dist_harm = result_csv.stem.split('_distractor')[0].split('_')[-1].title()
    if dist_harm == 'None':
        dist_harm = 'No Distractor'
    df['distractor_harmonicity'] = dist_harm
    df['target_word'] = df['true_word_int'].replace(cv_class_2_word)
    df['pred_word'] = df['pred_word_int'].replace(cv_class_2_word)
    df = pd.merge(df,
                 meta_df[cols_to_merge],
                 left_on=['target_word','orig_df_row_ix'], right_on=['word', 'expt_df_ix'], how='left')     
    dfs.append(df)

exp_2_df = pd.concat(dfs, axis=0, ignore_index=True)
exp_2_df['confusions'] = (exp_2_df.pred_word == exp_2_df.same_sex_dist_1_word).astype('int') + (exp_2_df.pred_word == exp_2_df.diff_sex_dist_1_word).astype('int')
exp_2_df['accuracy'] = (exp_2_df.pred_word == exp_2_df.target_word).astype('int')
# add adjusted accuracy and confusions 
pred_words = exp_2_df.pred_word.values
target_words = exp_2_df.word.values
target_transcripts = exp_2_df.target_transcripts.values
same_sex_distractor_words = exp_2_df.same_sex_dist_1_word.values
diff_sex_distractor_words = exp_2_df.diff_sex_dist_1_word.values
same_sex_distractor_transcripts = exp_2_df.same_sex_distractor_1_transcripts.values
diff_sex_distractor_transcripts = exp_2_df.diff_sex_distractor_1_transcripts.values


adjusted_acc = np.array([int(pred_word in target_transcript or pred_word == target_word)
                            if not isinstance(target_transcript, float) else np.nan
                            for pred_word, target_word, target_transcript in zip(pred_words, target_words, target_transcripts)
                            ])

adjusted_confs = np.array([int(pred_word in same_sex_transcript or pred_word in diff_sex_transcript or pred_word == same_sex_word or pred_word == diff_sex_word)
                            if not (isinstance(same_sex_transcript, float) and isinstance(diff_sex_transcript, float)) else np.nan
                            for pred_word, same_sex_word, diff_sex_word, same_sex_transcript, diff_sex_transcript in zip(pred_words, same_sex_distractor_words, diff_sex_distractor_words,  same_sex_distractor_transcripts, diff_sex_distractor_transcripts)
                            ])

exp_2_df['adjusted_accuracy'] = adjusted_acc
exp_2_df['adjusted_confusions'] = adjusted_confs



def bootstrap_sem(data, n_bootstraps=1000):
    bootstrapped_means = np.zeros(n_bootstraps)
    n = len(data)
    for i in range(n_bootstraps):
        bootstrapped_sample = np.random.choice(data, size=n, replace=True)
        bootstrapped_means[i] = bootstrapped_sample.mean()
    return bootstrapped_means.std()


# exp_2_df = (exp_2_df.groupby(['model', "target_harmonicity", 'distractor_harmonicity']).agg({
#                             'adjusted_accuracy':['mean'],
#                             'adjusted_confusions':['mean', 'count']})
#                      .reset_index())
exp_2_df_summary = []
for (model, target_harm, dist_harm), group in exp_2_df.groupby(['model', 'target_harmonicity', 'distractor_harmonicity']):
    accs = group.adjusted_accuracy.values
    confs = group.adjusted_confusions.values
    acc_sem = bootstrap_sem(accs)
    conf_sem = bootstrap_sem(confs)
    if 'early' in model:
        model = 'Early-only'
    elif 'late' in model:
        model = 'Late-only'
    elif 'control' in model:
        model = 'Baseline CNN'
    elif '50Hz' in model:
        model = '50Hz cutoff'
    elif 'backbone' in model:
        model = 'Computed-gain model'
    exp_2_df_summary.append(
                            {'model':model, 'target_harmonicity': target_harm, 'distractor_harmonicity': dist_harm, 
                             'accuracy': accs.mean(), 'accuracy_sem': acc_sem, 'confusions': confs.mean(), 'confusions_sem': conf_sem}
    ) 
exp_2_df = pd.DataFrame(exp_2_df_summary)
exp_2_df['N'] = 1
exp_2_df['group'] = exp_2_df['model']
exp_2_df.head()

exp_2_df['background_condition'] = exp_2_df['target_harmonicity'] + '_target_' + exp_2_df['distractor_harmonicity'] + '_distractor'
exp_2_df.drop(['target_harmonicity', 'distractor_harmonicity'], axis=1, inplace=True)

exp_2_df['snr'] = 0 # add snr column to match diotic_results
popham_conds_to_keep = ['Harmonic_target_Harmonic_distractor',
                        'Harmonic_target_No Distractor_distractor',
                        'Inharmonic_target_Inharmonic_distractor',
                        'Inharmonic_target_No Distractor_distractor',
                        'Whispered_target_No Distractor_distractor',
                        'Whispered_target_Whispered_distractor']

exp_2_df = exp_2_df[exp_2_df['background_condition'].isin(popham_conds_to_keep)].reset_index(drop=True)
exp_2_df['group']= "Norm before gain model"
exp_2_df.rename(columns={'accuracy_sem': 'acc_sem', 'confusions_sem': 'conf_sem'}, inplace=True)


12


## Experiment 6

In [ ]:
room_manifest = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/manifest_room.pdpkl')
# for tuple_ in room_manifest.itertuples():
#     print(tuple_)
#     break
room_material_map = {}
for row in room_manifest.itertuples():
    wall1 = row.material_x0[0].split(',')[0]
    wall2 = row.material_y1[0].split(',')[0]
    if wall1 == wall2:
        wall_str = f"{wall1} walls"
    else:
        wall_str = f"{wall1} and {wall2} walls"

    floor = row.material_z0[0].split(',')[0]
    ceiling = row.material_z1[0].split(',')[0]
    name_str = f"{wall_str} \n {floor} floor \n {ceiling} ceiling"

    if row.index_room in [5, 6]:
        name_str = 'Alternate speaker room'
    if row.index_room in [6, 8]:
        name_str += ' \n head rotated'
    else:
        head_rotated = ''
    if 'Anechoic' in name_str:
        name_str = "Anechoic"
    room_material_map[row.index_room] = name_str
# room_material_map = {row.index_room: f"{row.material_x0[0].split(',')[0]} and \n {row.material_y1[0].split(',')[0]} walls \n {row.material_z0[0].split(',')[0]} floor \n {row.material_z1[0].split(',')[0]} ceiling" for row in room_manifest.itertuples()}
# room_material_map[5] = 'standard speaker room'
room_material_map

{0: 'Anechoic',
 1: 'Wood panelling on glass fiber blanket walls \n Carpet on foam rubber padding floor \n Highly absorptive panels ceiling',
 2: 'Brick walls \n Wood parquet on concrete floor \n Plaster ceiling',
 3: 'Fiberglass wall treatment walls \n Linoleum floor \n Acoustic tiles ceiling',
 4: 'Concrete walls \n Linoleum floor \n Acoustic tiles ceiling',
 5: 'Alternate speaker room',
 6: 'Alternate speaker room \n head rotated',
 7: 'Fiberglass wall treatment walls \n Linoleum floor \n Acoustic tiles ceiling',
 8: 'Fiberglass wall treatment walls \n Linoleum floor \n Acoustic tiles ceiling \n head rotated'}

In [12]:

manifest_path = "binaural_test_manifests/human_array_exmpt_sim_v02_only_human_locs_w_noise_min_reverb_mit_room_v02.pkl"
with open(manifest_path, "rb") as f: 
    manifest = pickle.load(f)
manifest_df  = pd.DataFrame(manifest.values())
spkr_room_manifest = pd.read_pickle('/om2/user/imgriff/spatial_audio_pipeline/assets/brir/mit_bldg46room1004_min_reverb/manifest_room.pdpkl')



output_paths = list(Path(f"binaural_eval/simulate_2024_human_threshold_experiment_v02_30_dB_pink_noise_min_verb_mit46_1004").glob("*/*post*.pkl"))
# output_paths = list(Path(f"binaural_eval/simulate_2024_human_threshold_experiment_v02_cue_noise").glob("*/*.pkl"))
stim_manifest_df = pd.read_pickle('/om/user/imgriff/datasets/human_word_rec_SWC_2024/full_cue_target_distractor_df_w_meta.pdpkl')


word_class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 
ix_to_word = {v:k for k,v in word_class_map.items()}

remap_azim = lambda azim: 360 - azim if azim > 180 else 0 - azim 

results_dfs = []

gend_map = {True:'same', False:'different'}

for path in output_paths:

    res_dict = pickle.load(open(path, 'rb'))
    to_track = {key:val for key,val in res_dict.items() if key != 'textures'}

    df = pd.DataFrame.from_dict(to_track)
    # break
    df.rename(columns={"results": "accuracy"}, inplace=True)
    df['model'] = util_analysis.get_model_name(path.parent.stem)

    df.loc[df.index.values, ['word', 'distractor_word','sex_cond']] = stim_manifest_df.loc[df.stim_ix_list.values, ['word', 'distractor_word','sex_cond']].values
    df['target_word'] = df['true_word_int'].map(ix_to_word)
    df['pred_word'] = df['preds'].map(ix_to_word)
    df['correct'] = (df['true_word_int'] == df['preds']).astype('int')
    df['str_confusions'] = df[['pred_word', 'distractor_word']].apply(lambda x: 1 if x.pred_word in x.distractor_word else 0, axis=1)
    ## Get SNR level from path
    if 'clean' in path.stem:
        snr = 'clean'
    else:
        snr = int(re.search('(-?\d+)_SNR', path.stem).group(0).strip('_SNR'))
    df['snr'] = snr 
    ## Get texture leve lfrom path 
    if "no_texture" in path.stem:
        texture_level = 'no_texture'
    
    elif 'texture' in path.stem: 
        texture_level = re.search('(-?\d+)dB_bg_texture',path.stem).group(0).split('dB')[0]
    elif 'pink_noise' in path.stem:
        texture_level = re.search('(-?\d+)dB_bg_pink_noise',path.stem).group(0).split('dB')[0]


    df['texture_level'] = texture_level
    df["target_azim"], df["target_elev"] = path.stem.split('target_loc_')[1].split('_distract_loc_')[0].split('_')
    df["distractor_azim"], df["distractor_elev"] = path.stem.split('_distract_loc_')[1].split('_')[:2]
    # # map azim to 0-180
    df['target_azim'] = df['target_azim'].astype(int).apply(remap_azim)
    df['distractor_azim'] = df['distractor_azim'].astype(int).apply(remap_azim)
    df['target_elev'] = df['target_elev'].astype(int)
    df['distractor_elev'] = df['distractor_elev'].astype(int)
    df['room_ix'] = int(re.search('room(-?\d+)', path.stem).group(0).strip('room'))
    df['room_type'] = re.search('SNR_(.*?)_room', path.stem).group(0).split('SNR_')[-1].split("_room")[0]
    df['n_distractors'] = 1 if '1_distractor' in path.stem else 2
    df['test_set'] = 'all_stim' if 'all_stim' in path.stem else 'subset'
    results_dfs.append(df)

results = pd.concat(results_dfs)

print(results.model.value_counts())

results.loc[results.room_type.str.contains('eval'), 'room_str'] = results.loc[results.room_type == 'eval', 'room_ix'].map(room_material_map)

results.loc[results.room_type.str.contains('mitb46'), 'room_str'] = 'Normal speaker array'
results.loc[results.room_type.str.contains('mitb46'), 'room_ix'] = 9 # use n from diff room notebook

results.loc[results.room_type.str.contains('reverb'), 'room_str'] = 'Min. reverb speaker array'
results.loc[results.room_type.str.contains('reverb'), 'room_ix'] = 10 # use n from diff room notebook


grouped_model_results = results.groupby(['model', 'target_azim', 'target_elev', 'distractor_azim', 'test_set',  'texture_level',
                                   'distractor_elev', 'sex_cond', 'snr', 'n_distractors', 'room_str']).agg({'accuracy':['mean', 'sem'],
                                                                                                                     'confusions':['mean', 'sem']}).reset_index()
# flatten multiindex
grouped_model_results.columns = ['_'.join(col).strip() for col in grouped_model_results.columns.values]
# remove trailing underscore
grouped_model_results.columns = [col[:-1] if col[-1] == '_' else col for col in grouped_model_results.columns.values]


grouped_model_results['azim_delta'] = (grouped_model_results['distractor_azim'] - grouped_model_results['target_azim']).abs()
grouped_model_results['elev_delta'] = (grouped_model_results['distractor_elev'] - grouped_model_results['target_elev']).abs()

grouped_model_results['distractor_elev_delta'] = (grouped_model_results['target_elev'] - grouped_model_results['distractor_elev']).abs()
exp_6_df = grouped_model_results[grouped_model_results.snr.isin([6,3, 0, -3, -6, -9,])].copy()

exp_6_df = exp_6_df.groupby(['model', 'snr', 'azim_delta', 'elev_delta']).agg({'accuracy_mean':['mean', 'sem'], "confusions_mean":['mean', 'sem']}).reset_index()
exp_6_df.columns = ['model', 'snr', 'azim_delta', 'elev_delta', 'accuracy', 'accuracy_sem', 'confusions', 'confusions_sem']
exp_6_df.loc[exp_6_df.model.str.contains('post'), 'group'] = 'Norm before gain model'
exp_6_df['background_condition'] = exp_6_df['azim_delta'].astype('str') + ' azim delta ' + exp_6_df['elev_delta'].astype('str') + ' elev delta' 
exp_6_df.rename(columns={'accuracy_sem': 'acc_sem', 'confusions_sem': 'conf_sem'}, inplace=True)



model
word_task_v10_gain_post_norm_config    117120
Name: count, dtype: int64


# Experiment 7

In [13]:
# Path to results 

cols_to_merge = ['stim_name',
                 'word',
                 'sex_cond',
                 'target_transcripts',
                 'distractor_transcripts',
                 'distractor_word',

]

# output_paths = list(Path("binaural_eval/word_task_voice_loc_cue_only_v04").glob("*.pkl")) old path for more locations
import pickle
# match human pilot conditions

output_paths = list(Path(f"binaural_eval/sim_azim_spotlight_v02_min_reverb_room1004_30dB_pink_noise_bg/").glob("*/*post*.pkl"))

word_class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 
ix_to_word = {v:k for k,v in word_class_map.items()}

remap_azim = lambda azim: 360 - azim if azim > 180 else 0 - azim 

results_dfs = []

gend_map = {True:'same', False:'diff'}

for path in output_paths:
    try:
        df = pd.DataFrame(pd.read_pickle(path))
    
        df['response'] = df['preds'].replace(ix_to_word)
        df['true_word'] = df['true_word_int'].replace(ix_to_word)
        df.rename(columns={"results": "accuracy"}, inplace=True)

        df['target_azim'] = int(re.search('target_loc_(-?\d+)', path.stem).group(0).strip('target_loc_'))
        df['distractor_azim'] = int(re.search('distract_loc_(-?\d+)', path.stem).group(0).strip('distract_loc_'))
        df['target_azim'] = df['target_azim'].apply(remap_azim)
        df['distractor_azim'] = df['distractor_azim'].apply(remap_azim)
        df['model'] = path.parent.stem
        results_dfs.append(df)
    except Exception as error:
        print(error)
        print(path)
        continue


exp_7_df = pd.concat(results_dfs)
exp_7_df['target_azim'] = exp_7_df['target_azim'].abs()
exp_7_df['distractor_azim'] = exp_7_df['distractor_azim'].abs()
exp_7_df['azim_delta'] = np.abs(exp_7_df['distractor_azim'] - exp_7_df['target_azim'])
exp_7_df = exp_7_df[exp_7_df['azim_delta'].isin([0, 10, 30, 90])]
exp_7_df['group'] = exp_7_df['model']
exp_7_df.rename(columns={'target_azim':'Target azimuth'}, inplace=True)
exp_7_df.dropna(axis=1, inplace=True)
exp_7_df['azim_delta'] = exp_7_df['azim_delta'].astype(int)
exp_7_df['Target azimuth'] = exp_7_df['Target azimuth'].astype(int)


exp_7_df = exp_7_df.groupby(['model','Target azimuth', 'azim_delta']).agg({'accuracy': ['mean', 'sem'], 'confusions':['mean', 'sem']}).reset_index()
exp_7_df.columns = ['model', 'target_azim', 'azim_delta', 'accuracy', 'acc_sem', 'confusions', 'conf_sem']
exp_7_df['background_condition'] = exp_7_df['target_azim'].astype('str') + ' target azim ' + exp_7_df['azim_delta'].astype('str') + ' azim delta'
exp_7_df['snr'] = 0 


In [ ]:
print(exp_7_df.model.value_counts())

model
word_task_v10_gain_post_norm_config    8
Name: count, dtype: int64


## Merge experiment results 


In [ ]:
norm_then_gain_df = pd.concat([exp_1_df, exp_2_df, exp_6_df, exp_7_df], axis=0, ignore_index=True)
norm_then_gain_df['group'] = "Norm before gain model"
norm_then_gain_df['model'] = "Norm before gain model"
norm_then_gain_df.head()

,group,model,snr,background_condition,accuracy,acc_sem,confusions,conf_sem,N,azim_delta,elev_delta,target_azim
0,Norm before gain model,Norm before gain model,-9,1-talker,0.122439,0.007549,0.273566,0.010074,1.0,NaN,NaN,NaN
1,Norm before gain model,Norm before gain model,-9,2-talker,0.036885,0.006224,0.199795,0.012664,1.0,NaN,NaN,NaN
2,Norm before gain model,Norm before gain model,-9,4-talker,0.023566,0.004681,0.056352,0.007115,1.0,NaN,NaN,NaN
3,Norm before gain model,Norm before gain model,-9,babble,0.038934,0.006236,0.004098,0.002030,1.0,NaN,NaN,NaN
4,Norm before gain model,Norm before gain model,-9,music,0.268443,0.013954,0.001025,0.001035,1.0,NaN,NaN,NaN


#### Save results

In [ ]:
results_dir = Path('final_results_to_share')
outfile = results_dir / 'norm_then_gain_model_summary_all_experiments.csv'
to_save = norm_then_gain_df.copy()
to_save = to_save[['group', 'model', 'snr', 'background_condition', 'accuracy', 'acc_sem', 'confusions', 'conf_sem', 'N']]
# to_save.to_csv(outfile, index=False)